In [ ]:
# RF approach

# plan is to do a random forest on the features (using PCA) then the probabilties of the grouping is then fed into a hueristic a* search for the optimal grouping

# side note: since there is game theory (from the guessing information given), the search will be using live feedback to update the random forest

In [3]:
# convert existing word encoding using sentence transformers
from sentence_transformers import SentenceTransformer
import numpy as np
import nltk
from nltk import pos_tag

# Ensure we have the NLTK grammar tagger
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

class DynamicFeatureExtractor:
    def __init__(self):
        print("Initializing HuggingFace MiniLM...")
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.vowels = set('aeiouy')
        self.pos_categories = ['NN', 'NNS', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'JJ', 'RB']
        print("Dynamic Extractor Ready!")

    def get_vector(self, word):
        """Takes ANY word and generates a dynamic 396-dimension vector."""
        word = str(word).strip().lower()
        
        # 1. MiniLM Subword Vector (384 dims)
        semantic_vec = self.model.encode(word)
        
        # 2. Word Length (1 dim)
        length_vec = np.array([len(word) / 15.0]) 
        
        # 3. Vowel Ratio (1 dim)
        num_vowels = sum(1 for char in word if char in self.vowels)
        vowel_ratio = np.array([num_vowels / max(1, len(word))])
        
        # 4. Part of Speech One-Hot (10 dims)
        tag = pos_tag([word])[0][1] 
        pos_vec = np.zeros(len(self.pos_categories))
        if tag in self.pos_categories:
            pos_vec[self.pos_categories.index(tag)] = 1.0
            
        # Glue them together: 384 + 1 + 1 + 10 = 396 Dimensions
        enriched_vec = np.concatenate([
            semantic_vec, 
            length_vec, 
            vowel_ratio, 
            pos_vec
        ]).astype(np.float32)
        
        return enriched_vec

extractor = DynamicFeatureExtractor()

# Test an out-of-vocabulary slang word
test_word = "skibidi"
vector = extractor.get_vector(test_word)

print(f"\nSuccessfully generated dynamic vector for '{test_word}'!")
print(f"Vector Shape: {vector.shape}")

Initializing HuggingFace MiniLM...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dynamic Extractor Ready!

Successfully generated dynamic vector for 'skibidi'!
Vector Shape: (396,)


In [4]:
# Statistical aggregation for random forest

import numpy as np
import pandas as pd
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from itertools import combinations
from numpy.linalg import norm

# 1. Feature Aggregation Function
def get_group_features(words, extractor):
    """Compresses 4 words into a razor-sharp, 6-dimensional statistical array."""
    # 1. Get the raw 396-dim vectors from your extractor
    vectors = [extractor.get_vector(w) for w in words]
    
    # 2. Calculate every possible pair-wise comparison (6 pairs total for 4 words)
    similarities = []
    for vec1, vec2 in combinations(vectors, 2):
        n1, n2 = norm(vec1), norm(vec2)
        if n1 > 0 and n2 > 0:
            sim = np.dot(vec1, vec2) / (n1 * n2)
            similarities.append(float(sim))
        else:
            similarities.append(0.0)
            
    # 3. Extract the Semantic Stats (The "Logic" of the group)
    mean_sim = np.mean(similarities)
    min_sim = np.min(similarities)   # Catches the "Odd one out" fakes!
    max_sim = np.max(similarities)
    var_sim = np.var(similarities)
    
    # 4. Extract Structural Stats (Using dimensions 384 and 385 from your extractor)
    # Dim 384 is length, Dim 385 is vowel ratio
    lengths = [v[384] for v in vectors]
    vowels = [v[385] for v in vectors]
    
    var_length = np.var(lengths)
    var_vowels = np.var(vowels)
    
    # Glue them together into just 6 powerful dimensions
    return np.array([
        mean_sim, 
        min_sim, 
        max_sim, 
        var_sim, 
        var_length, 
        var_vowels
    ], dtype=np.float32)

print("Loading pre-labeled dataset...")
df = pd.read_csv("connections_training_dataset.csv") 

X = []
y = []

print("Extracting features from pre-labeled data...")
for _, row in df.iterrows():
    # Grab the 4 words for this row
    words = [
        str(row['word1']).strip().lower(), 
        str(row['word2']).strip().lower(), 
        str(row['word3']).strip().lower(), 
        str(row['word4']).strip().lower()
    ]
    
    # Grab your exact pre-made label (assuming 1 = valid, 0 = invalid)
    label = int(row['label'])
    
    # Ensure there are no duplicate/NaN errors in the row before passing it to the model
    if len(set(words)) == 4:
        X.append(get_group_features(words, extractor))
        y.append(label)

X = np.array(X)
y = np.array(y)

print(f"Dataset generated! Total samples: {len(X)}")
print(f"Positive samples: {sum(y == 1)}")
print(f"Negative samples: {sum(y == 0)}")

# 3. Train the Random Forest
print("Splitting data and training Random Forest...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 100 trees for 793 dimensions
rf_model = RandomForestClassifier(
    n_estimators=200,        # More trees
    max_depth=20,            # Let it think a little deeper
    class_weight='balanced', 
    random_state=42, 
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# 4. Evaluate
print("\n--- Model Evaluation (Custom Threshold 0.30) ---")
# Get the raw probability of being a '1' (Valid Category)
y_probs = rf_model.predict_proba(X_test)[:, 1] 

# Tell it to guess '1' if it is even 30% confident
custom_threshold = 0.30
y_pred_custom = (y_probs >= custom_threshold).astype(int)

print(classification_report(y_test, y_pred_custom))

# Save 
with open("connections_rf_model.pkl", "wb") as f:
    pickle.dump(rf_model, f)
print("\nModel saved as 'connections_rf_model.pkl'")

Loading pre-labeled dataset...
Extracting features from pre-labeled data...


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

# Get the raw probabilities again
y_probs = rf_model.predict_proba(X_test)[:, 1]

# Calculate precision, recall, and thresholds
precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)

# Calculate the F1 score for every single threshold
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10) # 1e-10 prevents divide by zero

# Find the index of the highest F1 score
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]
print(f"\nMathematically Optimal Threshold: {optimal_threshold:.3f}")

# Evaluate using the optimal threshold
y_pred_optimal = (y_probs >= optimal_threshold).astype(int)
print("\n--- Evaluation at Optimal Threshold ---")
print(classification_report(y_test, y_pred_optimal))


🎯 Mathematically Optimal Threshold: 0.203

--- Evaluation at Optimal Threshold ---
              precision    recall  f1-score   support

           0       0.78      0.24      0.36      1455
           1       0.35      0.86      0.50       708

    accuracy                           0.44      2163
   macro avg       0.57      0.55      0.43      2163
weighted avg       0.64      0.44      0.41      2163



In [ ]:
import gymnasium as gym
from gymnasium import spaces
import pandas as pd
import random

class ConnectionsEnv(gym.Env):
    def __init__(self, csv_path="connections_training_dataset.csv"):
        super(ConnectionsEnv, self).__init__()
        
        # Load dataset and filter ONLY the real categories (label == 1) to build the board
        self.df = pd.read_csv(csv_path)
        if 'label' in self.df.columns:
            self.valid_groups_df = self.df[self.df['label'] == 1]
        else:
            self.valid_groups_df = self.df
            
        # The UI Controller: 16 buttons
        self.action_space = spaces.Discrete(16)
        # Dummy observation space since the Detective Agent reads the board directly
        self.observation_space = spaces.Discrete(1) 
        
        self.current_board = []
        self.solution_groups = []
        self.remaining_indices = []
        self.lives = 4
        self.previous_guesses = set()
        self.current_selection = [] 
        
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        # Build a valid 16-word board
        while True:
            sampled_rows = self.valid_groups_df.sample(4)
            self.solution_groups = []
            words_pool = []
            valid_board = True
            
            for _, row in sampled_rows.iterrows():
                group = {str(row['word1']).strip().lower(), 
                         str(row['word2']).strip().lower(), 
                         str(row['word3']).strip().lower(), 
                         str(row['word4']).strip().lower()}
                
                # Ensure no blank cells or duplicate words in a single row
                if len(group) != 4:
                    valid_board = False
                    break
                    
                self.solution_groups.append(group)
                words_pool.extend(list(group))
                
            # Ensure all 16 words on the board are entirely unique
            if valid_board and len(set(words_pool)) == 16:
                break
                
        # Shuffle the board
        random.shuffle(words_pool)
        self.current_board = words_pool
        self.remaining_indices = list(range(16))
        self.lives = 4
        self.previous_guesses = set()
        self.current_selection = []
        
        return 0, {}
        
    def step(self, action):
        # 1. Invalid clicks (Removed words or already selected words)
        if action not in self.remaining_indices or action in self.current_selection:
            return 0, -1, False, False, {"status": "Invalid click"}
            
        # 2. Add to buffer
        self.current_selection.append(action)
        
        # 3. Waiting for 4 words
        if len(self.current_selection) < 4:
            return 0, 0, False, False, {"status": "Selecting"}
            
        # 4. We have 4 words! Evaluate the group.
        guess_indices = frozenset(self.current_selection)
        self.current_selection = []  # Clear buffer
        
        # Prevent immortal loop
        if guess_indices in self.previous_guesses:
            self.lives -= 1
            if self.lives <= 0:
                return 0, -15, True, False, {"status": "Game Over"}
            return 0, -10, False, False, {"status": "Repeated guess"}
            
        self.previous_guesses.add(guess_indices)
        guessed_words = {self.current_board[i] for i in guess_indices}
        
        terminated = False
        info = {}
        max_overlap = 0
        matched_group = None
        
        for group in self.solution_groups:
            overlap = len(guessed_words.intersection(group))
            if overlap > max_overlap:
                max_overlap = overlap
                matched_group = group

        # Feedback Logic
        if max_overlap == 4:
            self.remaining_indices = [i for i in self.remaining_indices if i not in guess_indices]
            self.solution_groups.remove(matched_group)
            info["status"] = "Correct!"
            
            if len(self.remaining_indices) == 0:
                terminated = True
                info["status"] = "Game Won!"
                
        elif max_overlap == 3:
            self.lives -= 1
            info["status"] = "One Away!"
            
        elif max_overlap == 2:
            self.lives -= 1
            info["status"] = "Two Away!"
            
        else:
            self.lives -= 1
            info["status"] = "Incorrect."
            
        if self.lives <= 0 and not terminated:
            terminated = True
            info["status"] = "Game Over"
            
        return 0, 0, terminated, False, info
    
    def load_custom_board(self, custom_groups):
        """
        Bypasses the random reset and loads a specific 16-word board.
        Expects a list of 4 lists, each containing 4 words.
        """
        self.solution_groups = []
        words_pool = []
        
        for group in custom_groups:
            # Clean and lowercase the inputs just to be safe
            cleaned_group = {str(w).strip().lower() for w in group}
            self.solution_groups.append(cleaned_group)
            words_pool.extend(list(cleaned_group))
            
        if len(set(words_pool)) != 16:
            print("WARNING: Your custom board does not have exactly 16 unique words!")
            
        # Shuffle the board so the AI doesn't just read them in order
        random.shuffle(words_pool)
        self.current_board = words_pool
        self.remaining_indices = list(range(16))
        self.lives = 4
        self.previous_guesses = set()
        self.current_selection = []
        
        return 0, {}

In [2]:
import numpy as np
from itertools import combinations

class DetectiveAgent:
    def __init__(self, env, rf_model, extractor):
        self.env = env
        self.rf_model = rf_model
        self.extractor = extractor
        self.constraints = [] # Stores our Game Theory rules
        
    def score_combinations(self, remaining_words):
        """Generates combos, scores them greedily, then re-ranks the top picks with Lookahead."""
        print(f"Calculating probabilities for {len(remaining_words)} words...")
        combos = list(combinations(remaining_words, 4))
        valid_combos = []
        
        # 1. APPLY CONSTRAINTS
        for combo in combos:
            combo_set = set(combo)
            is_valid = True
            for constraint_type, target_set in self.constraints:
                overlap = len(combo_set.intersection(target_set))
                if constraint_type == "one_away" and overlap != 3:
                    is_valid = False
                    break
                elif constraint_type == "incorrect" and overlap >= 3:
                    is_valid = False
                    break
            if is_valid:
                valid_combos.append(combo)
                
        if not valid_combos:
            print("WARNING: Constraints are too tight, wiping memory and guessing blind!")
            self.constraints = []
            valid_combos = combos

        # 2. PASS 1: GREEDY SEARCH
        features = [get_group_features(c, self.extractor) for c in valid_combos]
        features_array = np.array(features)
        probs = self.rf_model.predict_proba(features_array)[:, 1]
        
        # Sort and take the Top 15 to prevent CPU meltdown
        ranked_greedy = sorted(zip(valid_combos, probs), key=lambda x: x[1], reverse=True)
        candidates = ranked_greedy[:15] 
        
        # 3. PASS 2: THE LOOKAHEAD (Holistic Check)
        final_rankings = []
        for combo, base_score in candidates:
            leftovers = [w for w in remaining_words if w not in combo]
            
            if len(leftovers) < 4:
                # If this guess clears the board, it's a perfect board state!
                lookahead_score = 1.0 
            else:
                # Find the single BEST group we can make with the leftovers
                leftover_combos = list(combinations(leftovers, 4))
                leftover_features = [get_group_features(c, self.extractor) for c in leftover_combos]
                leftover_probs = self.rf_model.predict_proba(np.array(leftover_features))[:, 1]
                lookahead_score = np.max(leftover_probs) 
                
            # Blend the scores: 60% current guess, 40% leftover board health
            holistic_score = (base_score * 0.6) + (lookahead_score * 0.4)
            final_rankings.append((combo, holistic_score))
            
        # Return the final re-sorted list
        return sorted(final_rankings, key=lambda x: x[1], reverse=True)

    def play(self, custom_groups=None):
        if custom_groups:
            obs, info = self.env.load_custom_board(custom_groups)
        else:
            obs, info = self.env.reset()
        done = False
        turn_count = 1
        
        print("="*50)
        print("AI STARTING GAME")
        print("="*50)
        
        while not done:
            remaining_words = [self.env.current_board[i] for i in self.env.remaining_indices]
            
            print(f"\n--- Turn {turn_count} (Lives: {self.env.lives}) ---")
            ranked_combos = self.score_combinations(remaining_words)
            
            # Pick the #1 mathematically best guess
            best_guess, best_score = ranked_combos[0]
            print(f"Top Guess: {best_guess} (Confidence: {best_score:.2f})")
            
            # Translate words back to button indices for the environment
            guess_indices = [self.env.current_board.index(w) for w in best_guess]
            
            # Push the 4 buttons!
            for idx in guess_indices:
                obs, reward, terminated, truncated, info = self.env.step(idx)
                
            status = info.get("status")
            print(f">> Result: {status}")
            
            # THE GAME THEORY
            guess_set = set(best_guess)
            if status == "Correct!" or status == "Game Won!":
                # The words are removed from the board, no constraints needed.
                pass
            elif status == "One Away!":
                print(">> Pivot: Activating Hard Constraint (Must share 3 words)")
                self.constraints.append(("one_away", guess_set))
            else:
                # "Two Away" or "Incorrect"
                print(">> Pivot: Activating Soft Constraint (Avoiding this cluster)")
                self.constraints.append(("incorrect", guess_set))

            if terminated:
                done = True
                if status == "Game Won!":
                    print("\nTHE DETECTIVE BEAT THE GAME!")
                else:
                    print("\nTHE DETECTIVE RAN OUT OF LIVES.")
            
            turn_count += 1


# 1.custom board (The Answer Key)
test_board = [
    ["chin", "ear", "hair", "lip"],                 # Starts with body parts
    ["bow", "line", "tie", "zip"],                  # ___ tie
    ["drill", "exercise", "practice", "routine"],   # Semantic synonyms
    ["barbie", "bratz", "cabbage patch", "troll"]   # Vintage dolls
]

# 2. Initialize the environment and agent
env = ConnectionsEnv()
agent = DetectiveAgent(env, rf_model, extractor)

# 3. Pass the custom board directly into the play function!
agent.play(custom_groups=test_board)

NameError: name 'ConnectionsEnv' is not defined